# Fuego contra Fuego


Instalamos, cargamos y seteamos el entorno

In [ ]:
#%pip install optuna


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 6.1 MB/s eta 0:00:00


In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from tqdm.notebook import tqdm

from sklearn.tree import DecisionTreeClassifier, plot_tree,  _tree
from sklearn.model_selection import train_test_split
from sklearn.model_selection import ShuffleSplit, StratifiedShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer

from joblib import Parallel, delayed

import optuna
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_slice, plot_contour

from time import time

import pickle


c:\Users\carof\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## DATA BASE

In [2]:
base_path = "/Users/carof/Downloads/DENICOLAY/"
dataset_path = base_path 
modelos_path = base_path + 'modelos/'
db_path = base_path + 'data/'
dataset_file = 'competencia_01.csv'

ganancia_acierto = 1_072_500
costo_estimulo = 27_500

mes_train = 202103
mes_test = 202105

# agregue sus semillas
semilla = 42
semillas = np.random.choice(1000000, size=50, replace=False).tolist()

data = pd.read_csv(dataset_path + dataset_file)

Seguimos trabajando con Marzo como entrenamiento y Mayo como test.

In [3]:
X = data[data['foto_mes'] == mes_train]
y = X['clase_ternaria']
X = X.drop(columns=['clase_ternaria'])

In [4]:
X_futuro = data[data['foto_mes'] == mes_test]
y_futuro = X_futuro['clase_ternaria']
X_futuro = X_futuro.drop(columns=['clase_ternaria'])

Y variaremos la forma de la función de ganancia, para poder ser utilizada de una forma más genérica.

In [ ]:
def ganancia_prob(y_hat, y, prop=1, class_index=1, threshold=0.025):
  @np.vectorize
  def ganancia_row(predicted, actual, threshold=0.025):
    return  (predicted >= threshold) * (ganancia_acierto if actual == "BAJA+2" else -costo_estimulo)

  return ganancia_row(y_hat[:,class_index], y).sum() / prop

Entrenemos y recordemos cuanto nos daba la ganancia en **Mayo**

Hablemos de ensemables, qué son?

* Un ensemble de modelos es una técnica donde se combinan múltiples modelos individuales para mejorar la precisión y robustez de las predicciones. La idea es que al combinar varios modelos, se pueden aprovechar las fortalezas de cada uno y reducir la posibilidad de errores que podría cometer un único modelo.

**Tipos de ensemble**:
 * **Bagging (Bootstrap Aggregating)**: Consiste en entrenar varios modelos base en diferentes subconjuntos del conjunto de datos de entrenamiento obtenidos mediante técnicas de remuestreo como el bootstrap y luego promediar sus predicciones. Ejemplo: **Random Forest**
 * **Boosting**: En esta técnica, los modelos se entrenan de manera secuencial. Cada modelo intenta corregir los errores cometidos por el modelo anterior. Ejemplo: AdaBoost y **Gradient Boosting**.
 * **Stacking**: En el stacking, se entrenan varios modelos y se combinan usando un "modelo meta". Las predicciones de los modelos base sirven como features para entrenar este modelo meta, que produce la predicción final.

* **Ventajas de usar ensemble de modelos**:
 * **Mejor rendimiento**: Al combinar modelos, generalmente se mejora la precisión en comparación con un solo modelo.
 * **Robustez**: Al integrar diferentes modelos, se mitiga el riesgo de que los errores de un modelo individual afecten gravemente la predicción final.




Pongamos foco en el **Random Forest**

Es un algoritmo de aprendizaje automático que funciona creando un conjunto de árboles de decisión. Para la creación de **árboles distintos** utiliza una técnica llamada bagging para crear múltiples subconjuntos del conjunto de datos de entrenamiento. Cada subconjunto se genera seleccionando al azar muestras del conjunto de datos original con reemplazo. No usa la totalidad de los datos de entrenamiento para cada conjunto. Los datos que quedan fueran son conocidos como **Out of Bag (oob)**

Para cada subconjunto, se construye un árbol de decisión. Sin embargo en cada nodo del árbol, **Random Forest** selecciona de forma aleatoria un grupo de variables y ajusta el árbol con esas variables. Este proceso ayuda a crear árboles que son menos correlacionados entre sí.

Cada árbol en el bosque se entrena de manera independiente usando su respectivo subconjunto de datos. Lueago, para una nueva observación, cada árbol realiza una predicción. El Random Forest luego combina las predicciones de todos los árboles para hacer una predicción final, devolviendo el promedio de las probabilidades de cada árbol individual.



Como desde la clase pasada solo personas más inteligentes, no vamos a empezar a probar **Random Forest** simples. Vamos a parametrizarlo desde el vamos.

Primero vamos a entender algunas limitaciones de la implementación:

El **Random Forest** no soporta nulos! Shame on you sklearn!.

Vamos a tener que imputar los datos. Discutamos entre todos forma de imputar los datos, mientras para salir del paso usamos la peor de todas.

In [6]:
imp_median = SimpleImputer(missing_values=np.nan, strategy='median')
Xi = imp_median.fit_transform(X)
Xif = imp_median.fit_transform(X_futuro)

Los parámetros que se pueden ajustar en el **rf** son

1. **n_estimators**: Número de árboles en el bosque.
2. **max_depth**: Profundidad máxima de los árboles.
3. **min_samples_split**: Número mínimo de muestras requeridas para dividir un nodo interno.
4. **min_samples_leaf**: Número mínimo de muestras requeridas para estar en un nodo hoja.
5. **max_features**: Número de features a usar en cada árbol. **sqrt** es una elección histórica.
6. **max_leaf_nodes**: Número máximo de nodos hoja en cada árbol.
7. **oob_score**: Indica si se usa la muestra fuera de bolsa (out-of-bag) para estimar la calidad del modelo. Para evitar hacer un **montecarlo-cross-validation** que se toma su tiempo, usaremos esta opción para buscar el mejor modelo. No es la mejor opción. Pero no es tan mala.
8. **n_jobs**: Siempre -1, para que use todos los cores presentes en 
9. **max_samples**: Fracción de los samples.

Finalmente nuestra función de optimización queda la siguiente forma:

In [13]:
def objective(trial):
    max_depth = trial.suggest_int('max_depth', 2, 32)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 2000)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 200)
    max_features = trial.suggest_float('max_features', 0.05, 0.7)

    model = RandomForestClassifier(
        n_estimators=100,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        max_samples=0.7,
        random_state=semillas[0],
        n_jobs=-1,
        oob_score=True
    )

    model.fit(Xi, y)

    return ganancia_prob(model.oob_decision_function_, y)

storage_name = "sqlite:////Users/carof/Downloads/DENICOLAY/optimization_rf.db"
study_name = "exp_401_random-forest-opt"

study = optuna.create_study(
    direction="maximize",
    study_name=study_name,
    storage=storage_name,
    load_if_exists=True,
)

[I 2026-09-07 16:58:06,910] A new study created in RDB with name: exp_401_random-forest-opt


### Primera optimizacion

In [ ]:
#study.optimize(objective, n_trials=100)

[I 2026-09-07 16:59:37,342] Trial 0 finished with value: 325902500.0 and parameters: {'max_depth': 30, 'min_samples_split': 1172, 'min_samples_leaf': 158, 'max_features': 0.47705155260923787}. Best is trial 0 with value: 325902500.0.
[I 2026-09-07 17:02:18,386] Trial 1 finished with value: 341495000.0 and parameters: {'max_depth': 14, 'min_samples_split': 490, 'min_samples_leaf': 169, 'max_features': 0.5804393957922185}. Best is trial 1 with value: 341495000.0.
[I 2026-09-07 17:13:26,320] Trial 2 finished with value: 337562500.0 and parameters: {'max_depth': 17, 'min_samples_split': 898, 'min_samples_leaf': 43, 'max_features': 0.5776010248361468}. Best is trial 1 with value: 341495000.0.
[I 2026-09-07 17:14:57,019] Trial 3 finished with value: 302885000.0 and parameters: {'max_depth': 5, 'min_samples_split': 152, 'min_samples_leaf': 121, 'max_features': 0.28844064612703657}. Best is trial 1 with value: 341495000.0.
[I 2026-09-07 17:18:22,171] Trial 4 finished with value: 346197500.0 an

In [8]:
# PARA CARGAR SI YA OPTIMICE
storage_name = "sqlite:////Users/carof/Downloads/DENICOLAY/optimization_rf.db"
study_name = "exp_401_random-forest-opt"

# Cargar el estudio existente
study = optuna.load_study(study_name=study_name, storage=storage_name)

# Ver los mejores resultados
print(f"Mejor valor de ganancia: {study.best_value}")
print(f"Mejores hiperparámetros: {study.best_params}")

Mejor valor de ganancia: 350432500.0
Mejores hiperparámetros: {'max_depth': 16, 'min_samples_split': 19, 'min_samples_leaf': 50, 'max_features': 0.30786715539121634}


Exploramos como fue la búsqueda de parámetros

In [15]:
optuna.visualization.plot_optimization_history(study)

In [16]:
plot_param_importances(study)

In [17]:
plot_slice(study)

In [18]:
plot_contour(study)

In [19]:
plot_contour(study, params=["max_depth", "min_samples_split"])

Ajustamos el mejor modelo a todo el conjunto de datos



In [20]:
model_rf = RandomForestClassifier(
        n_estimators=100,
        **study.best_params,
        max_samples=0.7,
        random_state=semillas[0],
        n_jobs=-1,
        oob_score=True
    )

model_rf.fit(Xi, y)


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",16
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",19
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",50
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",0.30786715539121634
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with sign

Guardamos el modelo, para no tener que optimizar cada vez que lo queramos usar.

In [ ]:
filename = 'exp_206_random_forest_model_100.sav'
#pickle.dump(model_rf, open(filename, 'wb')) # descomentar para guardar


Y lo cargamos para utilizarlo

In [11]:
model_rf = pickle.load(open(filename, 'rb'))
model_rf

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",16
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",19
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",50
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",0.30786715539121634
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with sign

Ahora vamos medir su ganancia sobre el dataset de **Mayo**

In [16]:
y_pred_rf = model_rf.predict_proba(Xif) # Xif era el Imputado Futuro
ganancias_rf_100 = ganancia_prob(y_pred_rf, y_futuro)
print(f"Ganancia de modelo RF: {ganancias_rf_100}")

Ganancia de modelo RF: 256300000.0


Frente a los 208.010.000 de mi mejor árbol es algo sustancialmente mejor.

Igualmente usamos poquitos estimadores. Tan solo unos 100. Suficientes como para que la optimización no tarde una eternidad, pero es muy poco. Deberíamos sumar unos cuantos más.

### RF 1000

In [24]:
model_rf_1000 = RandomForestClassifier(
        n_estimators=1000,
        **study.best_params,
        max_samples=0.7,
        random_state=semillas[0],
        n_jobs=-1,
        oob_score=True
    )

model_rf_1000.fit(Xi, y) # descomentar para entrenar

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",1000
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",16
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",19
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",50
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",0.30786715539121634
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with sig

In [25]:
filename_rf_1000 = 'exp_206_random_forest_model_1000.sav'
pickle.dump(model_rf_1000, open(filename_rf_1000, 'wb')) # descomentar para guardar

In [26]:
filename_rf_1000 = 'exp_206_random_forest_model_1000.sav'
model_rf_1000 = pickle.load(open(filename_rf_1000, 'rb'))
model_rf_1000

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",1000
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",16
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",19
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",50
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",0.30786715539121634
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with sig

Veamos si sumar 10 veces más estimadores tuvo algún efecto

In [27]:
y_pred_rf = model_rf_1000.predict_proba(Xif)
ganancias_rf_1000 = ganancia_prob(y_pred_rf, y_futuro)
print(f"Ganancia de modelo RF 1000: {ganancias_rf_1000}")
print(f"Mejora respecto de modelo RF 1000: {ganancias_rf_100 - ganancias_rf_1000}")

Ganancia de modelo RF 1000: 258445000.0
Mejora respecto de modelo RF 1000: -2145000.0


Mejoró un poco! Cada peso vale.



### Feature importance

Por último evaluamos cuales son las variables más importantes del modelo. ¿Habrá influido la imputación?

In [28]:
importances = model_rf.feature_importances_

features = X.columns
feat_importances = pd.DataFrame({'feature': features, 'importance': importances})
feat_importances = feat_importances.sort_values('importance', ascending=False)

feat_importances.head(25)


,feature,importance
107,ctrx_quarter,0.140490
22,mcuentas_saldo,0.070104
18,mcaja_ahorro,0.064515
11,mpasivos_margen,0.060336
33,mprestamos_personales,0.045252
8,mrentabilidad_annual,0.041054
16,mcuenta_corriente,0.038181
133,Visa_status,0.028948
28,mtarjeta_visa_consumo,0.024659
7,mrentabilidad,0.022076


Vamos a hacer un par de experimentos, que pasa si sacamos la variable más importante?

In [ ]:
most_important_feature = feat_importances.iloc[0]['feature']

X_df = pd.DataFrame(Xi, columns=X.columns)
X_futuro_df = pd.DataFrame(Xif, columns=X_futuro.columns)

X_no_best_feature = X_df.drop(columns=[most_important_feature])
X_futuro_no_best_feature = X_futuro_df.drop(columns=[most_important_feature])

model_rf_no_best_feature = RandomForestClassifier(
    n_estimators=100,
    **study.best_params,
    max_samples=0.7,
    random_state=semillas[0],
    n_jobs=-1,
    oob_score=True
)

print(f"Entrenando modelo RF sin '{most_important_feature}'...")
model_rf_no_best_feature.fit(X_no_best_feature, y)

y_pred_rf_no_best_feature = model_rf_no_best_feature.predict_proba(X_futuro_no_best_feature)

ganancia_rf_no_best_feature = ganancia_prob(y_pred_rf_no_best_feature, y_futuro)

Entrenando modelo RF sin 'ctrx_quarter'...
Ganancia del modelo RF sin 'ctrx_quarter': 249425000.0
Ganancia del modelo RF original: 256300000.0


In [30]:
print(f"Ganancia de modelo sin '{most_important_feature}' {ganancia_rf_no_best_feature}: ")
print(f"Diferencia respecto de modelo RF 1000: {ganancias_rf_100 - ganancia_rf_no_best_feature}") 

Ganancia de modelo sin 'ctrx_quarter' 249425000.0: 
Diferencia respecto de modelo RF 1000: 6875000.0


¿Qué paso? ¿Se ha derrumbado las ganancias frente a la ausencia del pilar de los árboles?

Una prueba más, que pasa si sumamos variables que son ruido puro.

In [31]:
X_df_from_Xi = pd.DataFrame(Xi, columns=X.columns)

np.random.seed(semilla)

random_features_train = pd.DataFrame({
    f'random_feature_{i}': np.random.rand(len(X_df_from_Xi)) for i in range(1, 6)
})

X_with_random_features = pd.concat([X_df_from_Xi, random_features_train], axis=1)

model_rf_new_features = RandomForestClassifier(
    n_estimators=100,
    **study.best_params,
    max_samples=0.7,
    random_state=semillas[0],
    n_jobs=-1,
    oob_score=True
)

model_rf_new_features.fit(X_with_random_features, y)

importances_new_features = model_rf_new_features.feature_importances_

features_new_features = X_with_random_features.columns
feat_importances_new_features = pd.DataFrame({'feature': features_new_features, 'importance': importances_new_features})
feat_importances_new_features = feat_importances_new_features.sort_values('importance', ascending=False)

display(feat_importances_new_features.head(25))

,feature,importance
107,ctrx_quarter,0.112575
11,mpasivos_margen,0.069912
22,mcuentas_saldo,0.064221
18,mcaja_ahorro,0.052660
33,mprestamos_personales,0.043998
8,mrentabilidad_annual,0.038392
16,mcuenta_corriente,0.037285
133,Visa_status,0.031915
28,mtarjeta_visa_consumo,0.024686
7,mrentabilidad,0.021234


Y una más, cual es el área bajo la curva en train?

In [ ]:
from sklearn.metrics import roc_auc_score

y_pred_rf_train = model_rf_1000.predict_proba(Xi)

ganancia_rf_train = ganancia_prob(y_pred_rf_train, y)
print(f"Ganancia en el conjunto de entrenamiento: {ganancia_rf_train}")
print(f"Ganancia en el conjunto de Mayo: {ganancias_rf_1000}")

y_train_numeric = (y == 'BAJA+2').astype(int) # dicotomizo
y_futuro_numeric = (y_futuro == 'BAJA+2').astype(int)

auc_train = roc_auc_score(y_train_numeric, y_pred_rf_train[:, 1])
print(f"Área bajo la curva (AUC) en el conjunto de entrenamiento: {auc_train}")

auc_mayo = roc_auc_score(y_futuro_numeric, y_pred_rf[:, 1])
print(f"Área bajo la curva (AUC) en el conjunto de Mayo: {auc_mayo}")

Ganancia en el conjunto de entrenamiento: 606512500.0
Ganancia en el conjunto de Mayo: 258445000.0
Área bajo la curva (AUC) en el conjunto de entrenamiento: 0.97180049967066
Área bajo la curva (AUC) en el conjunto de Mayo: 0.885930621841776


Cuanta diferencia entre conjuntos. Seguramente debemos descartar este algoritmo, ¿o no?

Pregunta:
* **¿Cómo sabe que un random forest es superior a otro?**
* **¿Cómo supera el random forest presentado?**

## Tarea:

* Mejore la parametrización del **rf**


### MEJORA OPTIMIZACION

Comparo su performance en test?
Como supero? creando nuevos features, optimizando

Cuando la consigna pide "mejorar la parametrización del Random Forest" tras haber corrido ya una optimización básica con Optuna, generalmente apunta a deficiencias en el diseño del espacio de búsqueda, en la estrategia de validación o en la configuración estructural del árbol.

**Revisar rangos y distribuciones de búsqueda en Optuna:**

Espacios lineales vs. logarítmicos: Hiperparámetros como min_samples_split (de 2 a 2000) o min_samples_leaf (de 1 a 200) suelen beneficiarse de distribuciones logarítmicas (trial.suggest_int('min_samples_split', 2, 2000, log=True)), ya que probar valores en escalas de magnitud (ej. 2, 10, 50, 200) explora mejor que una búsqueda uniforme.

Valores óptimos en los bordes: Si el mejor valor encontrado quedó pegado a uno de los extremos de tu rango (por ejemplo, max_features = 0.7 o max_depth = 32), el espacio de búsqueda estaba truncado y requiere ampliarse.

**Incorporar hiperparámetros faltantes:**

criterion: Evaluar "gini" frente a "entropy" / "log_loss".

max_samples: Dejaste fijo max_samples=0.7; optimizar este porcentaje de submuestreo (ej. entre 0.3 y 0.9) tiene un impacto directo en la varianza del ensamble.

class_weight: Si el problema tiene clases desbalanceadas, evaluar "balanced", "balanced_subsample" o ponderaciones personalizadas.

min_impurity_decrease: Controla la poda de divisiones que no aporten ganancia suficiente.

**Mejorar el esquema de validación y la función objetivo:**

Dependencia de oob_score: El cálculo OOB con 100 árboles puede tener alta varianza y sesgos si el dataset tiene estructura temporal o de grupos. Si se trata de series de tiempo o datos bancarios (común en problemas de optimización de ganancia), se suele requerir validación cruzada temporal o estratificada en lugar de OOB puro.

Número de árboles en la optimización: Optimizar con 100 árboles para luego entrenar con 1000 puede desplazar el punto óptimo de parámetros como max_features y min_samples_leaf, ya que un ensamble más grande tolera árboles más profundos y con mayor varianza individual.

**Aumentar el presupuesto de búsqueda:**

100 trials para un espacio continuo de 4 o más dimensiones puede ser insuficiente para que el muestreador bayesiano (TPE de Optuna) converja a un óptimo global robusto. Aumentar a 200–500 ensayos o cambiar el sampler (ej. TPESampler(multivariate=True)) mejora la exploración de correlaciones entre hiperparámetros.

Para usar validación cruzada en lugar de oob_score, utiliza StratifiedKFold (o KFold) para dividir los datos, entrena en los pliegues de entrenamiento y evalúa tu métrica ganancia_prob sobre las probabilidades predichas en cada conjunto de validación.

Desactiva oob_score=False (o quítalo) para ahorrar tiempo de cómputo innecesario:

In [34]:
# 5 folds estratificados
from sklearn.model_selection import StratifiedKFold
n_splits = 5
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=semillas[0])

def objective(trial):
    max_depth = trial.suggest_int('max_depth', 2, 32)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 500, log=True)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 200, log=True)
    max_features = trial.suggest_float('max_features', 0.05, 0.7)
    max_samples = trial.suggest_float('max_samples', 0.3, 1.0)

    fold_scores = []

    for train_idx, val_idx in cv.split(Xi, y):
        X_train = Xi.iloc[train_idx] if hasattr(Xi, 'iloc') else Xi[train_idx]
        y_train = y.iloc[train_idx] if hasattr(y, 'iloc') else y[train_idx]
        X_val = Xi.iloc[val_idx] if hasattr(Xi, 'iloc') else Xi[val_idx]
        y_val = y.iloc[val_idx] if hasattr(y, 'iloc') else y[val_idx]

        model = RandomForestClassifier(
            n_estimators=200,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
            max_samples=max_samples,
            random_state=semillas[0],
            n_jobs=-1,
            oob_score=False
        )

        model.fit(X_train, y_train)

        # Matriz completa de probabilidades (n_muestras, n_clases)
        y_val_probs = model.predict_proba(X_val)

        # Detectar la columna exacta de 'BAJA+2' dinámicamente
        target_class_idx = list(model.classes_).index("BAJA+2")

        # Se ajusta prop al tamaño del fold relativo al total
        fold_gain = ganancia_prob(
            y_hat=y_val_probs,
            y=y_val,
            prop=1 / n_splits,
            class_index=target_class_idx
        )
        fold_scores.append(fold_gain)

    return float(np.mean(fold_scores))

In [ ]:
# Ver los mejores resultados ANTERIORES
#print(f"Mejor valor de ganancia: {study.best_value}")
#print(f"Mejores hiperparámetros: {study.best_params}")

Mejor valor de ganancia: 350432500.0
Mejores hiperparámetros: {'max_depth': 16, 'min_samples_split': 19, 'min_samples_leaf': 50, 'max_features': 0.30786715539121634}


In [35]:
storage_name = "sqlite:////Users/carof/Downloads/DENICOLAY/optimization_rf.db"
study_name = "exp_401_random-forest-opt-vTarea"

study = optuna.create_study(
    direction="maximize",
    study_name=study_name,
    storage=storage_name,
    load_if_exists=True,
)

[I 2026-09-08 14:29:16,901] A new study created in RDB with name: exp_401_random-forest-opt-vTarea


In [ ]:
study.optimize(objective, n_trials=150)

[I 2026-09-08 14:37:19,192] Trial 0 finished with value: 311217500.0 and parameters: {'max_depth': 8, 'min_samples_split': 298, 'min_samples_leaf': 5, 'max_features': 0.5016464959948441, 'max_samples': 0.6720029296484965}. Best is trial 0 with value: 311217500.0.
[I 2026-09-08 14:51:22,055] Trial 1 finished with value: 297082500.0 and parameters: {'max_depth': 20, 'min_samples_split': 33, 'min_samples_leaf': 3, 'max_features': 0.3393453246702478, 'max_samples': 0.9978409801832846}. Best is trial 0 with value: 311217500.0.
[I 2026-09-08 15:03:52,354] Trial 2 finished with value: 339432500.0 and parameters: {'max_depth': 32, 'min_samples_split': 6, 'min_samples_leaf': 198, 'max_features': 0.6217546322766195, 'max_samples': 0.8226789400129564}. Best is trial 2 with value: 339432500.0.
[I 2026-09-08 15:08:00,413] Trial 3 finished with value: 324390000.0 and parameters: {'max_depth': 6, 'min_samples_split': 8, 'min_samples_leaf': 33, 'max_features': 0.3703942514393082, 'max_samples': 0.9689

: 

In [10]:
# PARA CARGAR SI YA OPTIMICE
storage_name = "sqlite:////Users/carof/Downloads/DENICOLAY/optimization_rf.db"
study_name = "exp_401_random-forest-opt-vTarea"

# Cargar el estudio existente
study = optuna.load_study(study_name=study_name, storage=storage_name)

# Ver los mejores resultados
print(f"Mejor valor de ganancia: {study.best_value}")
print(f"Mejores hiperparámetros: {study.best_params}")

Mejor valor de ganancia: 359810000.0
Mejores hiperparámetros: {'max_depth': 32, 'min_samples_split': 7, 'min_samples_leaf': 34, 'max_features': 0.305501478340253, 'max_samples': 0.969850310007658}


In [ ]:
# PARA CARGAR SI YA OPTIMICE # V Anterior para comparar
storage_name_anterior = "sqlite:////Users/carof/Downloads/DENICOLAY/optimization_rf.db"
study_name_anterior = "exp_401_random-forest-opt"

# Cargar el estudio existente
study_anterior = optuna.load_study(study_name=study_name_anterior, storage=storage_name_anterior)

# Ver los mejores resultados
print(f"Mejor valor de ganancia: {study_anterior.best_value}")
print(f"Mejores hiperparámetros: {study_anterior.best_params}")

Mejor valor de ganancia: 350432500.0
Mejores hiperparámetros: {'max_depth': 16, 'min_samples_split': 19, 'min_samples_leaf': 50, 'max_features': 0.30786715539121634}


In [11]:
study_anterior.best_value - study.best_value

-9377500.0

In [11]:
optuna.visualization.plot_optimization_history(study)

In [12]:
plot_param_importances(study)

In [13]:
plot_slice(study)

In [14]:
plot_contour(study)

In [13]:
model_rf_1000_vCaro = RandomForestClassifier(
        n_estimators=1000,
        **study.best_params,
        #max_samples=0.7,
        random_state=semillas[0],
        n_jobs=-1,
        oob_score=True
    )

model_rf_1000_vCaro.fit(Xi, y) # descomentar para entrenar

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",1000
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",32
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",7
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",34
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",0.305501478340253
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signat

In [14]:
filename_rf_1000_vCaro = 'exp_206_random_forest_model_1000_vCaro.sav'
pickle.dump(model_rf_1000_vCaro, open(filename_rf_1000_vCaro, 'wb')) # descomentar para guardar

In [15]:
y_pred_rf_vCaro = model_rf_1000_vCaro.predict_proba(Xif)
ganancias_rf_1000_vCaro = ganancia_prob(y_pred_rf_vCaro, y_futuro)


In [16]:
print(f"Ganancia de modelo RF 1000: {ganancias_rf_1000_vCaro}")


Ganancia de modelo RF 1000: 255915000.0


Ganancia de modelo RF 1000: 258445000.0
Mejora respecto de modelo RF 1000: -2145000.0

In [18]:
258445000 - 255915000

2530000

### RETRAIN Y OPTIMIZACION DESP DE FE

In [2]:
dataset_path = "/Users/carof/Downloads/DENICOLAY/"
parquet_file = f"{dataset_path}/competencia_01_fe.parquet"

data = pd.read_parquet(parquet_file)

# Verificación rápida
print(data.shape)
data.head()

(983061, 233)


,numero_de_cliente,foto_mes,active_quarter,cliente_vip,internet,cliente_edad,cliente_antiguedad,mrentabilidad,mrentabilidad_annual,mcomisiones,...,ctrx_quarter_ratio3,cconsumos_tc_ratio1,cconsumos_tc_ratio3,ctrx_quarter_ratio_avg3m,mcuentas_saldo_ratio_avg3m,ctrx_distancia_pico6m,mcuentas_saldo_distancia_pico6m,flag_cesacion_payroll,flag_vaciado_cuentas,flag_abandono_consumo_tc
0,12160484,202103,1,0,0,60,55,8101.55,3162.23,13399.50,...,NaN,NaN,NaN,1.0000,1.0000,1.0000,1.0000,0,0,0
1,12160484,202104,1,0,0,60,56,-3703.99,-1363.50,2091.38,...,NaN,0.8571,NaN,0.8889,-3.7943,0.8000,-0.6548,0,1,0
2,12160484,202105,1,0,0,60,57,-2660.01,-4707.72,2152.44,...,NaN,0.8333,NaN,0.7643,-1.7797,0.6154,-0.1285,0,0,0
3,12160484,202106,1,0,0,60,58,-1876.81,-6742.99,2438.26,...,0.5152,1.1000,0.7857,0.7920,-60.1410,0.5077,0.7461,0,0,0
4,12160484,202107,1,0,0,60,59,-2941.46,-8218.09,2929.35,...,0.5472,0.9091,0.8333,0.8317,-1.0464,0.4308,-0.1597,0,1,0


In [19]:
data.describe(include="all")

,numero_de_cliente,foto_mes,active_quarter,cliente_vip,internet,cliente_edad,cliente_antiguedad,mrentabilidad,mrentabilidad_annual,mcomisiones,...,ctrx_quarter_ratio3,cconsumos_tc_ratio1,cconsumos_tc_ratio3,ctrx_quarter_ratio_avg3m,mcuentas_saldo_ratio_avg3m,ctrx_distancia_pico6m,mcuentas_saldo_distancia_pico6m,flag_cesacion_payroll,flag_vaciado_cuentas,flag_abandono_consumo_tc
count,9.830610e+05,983061.000000,983061.000000,983061.000000,983061.000000,983061.000000,983061.000000,983061.000000,9.830610e+05,983061.000000,...,480791.000000,813334.000000,480791.000000,975224.000000,978024.000000,976060.000000,9.770770e+05,983061.000000,983061.000000,983061.000000
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,3.779833e+07,202105.506242,0.986954,0.003166,0.049117,46.884686,134.912766,2445.401757,2.484136e+04,1614.815041,...,1.107446,1.079329,1.194301,1.009897,2.172180,0.925760,-1.161873e+01,0.011059,0.047555,0.006817
std,1.586785e+07,1.707532,0.113472,0.056175,0.271282,12.987574,86.282920,8267.171556,6.822922e+04,3122.676446,...,1.030150,0.770395,1.135972,0.135415,794.057705,0.135264,3.548650e+03,0.104580,0.212822,0.082286
min,1.215985e+07,202103.000000,0.000000,0.000000,0.000000,18.000000,1.000000,-271488.390000,-9.380887e+05,-52407.900000,...,0.004600,0.004400,0.004400,0.000000,-43396.540500,0.000000,-2.006054e+06,0.000000,0.000000,0.000000
25%,2.549570e+07,202104.000000,1.000000,0.000000,0.000000,37.000000,63.000000,71.110000,1.884690e+03,366.730000,...,0.895300,0.900000,0.885700,0.971200,0.804600,0.898300,4.780000e-01,0.000000,0.000000,0.000000
50%,3.596115e+07,202106.000000,1.000000,0.000000,0.000000,45.000000,128.000000,1416.530000,1.334347e+04,981.750000,...,1.027800,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000e+00,0.000000,0.000000,0.000000
75%,4.905971e+07,202107.000000,1.000000,0.000000,0.000000,56.000000,190.000000,3304.210000,3.206790e+04,2181.850000,...,1.184000,1.142900,1.272700,1.050500,1.205000,1.000000,1.000000e+00,0.000000,0.000000,0.000000


In [21]:
pd.set_option("display.max_rows", None)
data.isnull().mean() * 100

numero_de_cliente                         0.000000
foto_mes                                  0.000000
active_quarter                            0.000000
cliente_vip                               0.000000
internet                                  0.000000
cliente_edad                              0.000000
cliente_antiguedad                        0.000000
mrentabilidad                             0.000000
mrentabilidad_annual                      0.000000
mcomisiones                               0.000000
mactivos_margen                           0.000000
mpasivos_margen                           0.000000
cproductos                                0.000000
tcuentas                                  0.000000
ccuenta_corriente                         0.000000
mcuenta_corriente_adicional               0.000000
mcuenta_corriente                         0.000000
ccaja_ahorro                              0.000000
mcaja_ahorro                              0.000000
mcaja_ahorro_adicional         

Claramente los lag6 son inutiles

Hacemos algo mas de preprocessing

In [3]:
# Dummy para Master: 1 si tiene dato, 0 si es NaN
data["dummy_Master_mora"] = data["Master_Finiciomora"].notna().astype(int)

# Dummy para Visa: 1 si tiene dato, 0 si es NaN
data["dummy_Visa_mora"] = data["Visa_Finiciomora"].notna().astype(int)

In [4]:
# Cantidad y porcentaje de nulos en el DataFrame (o en tu train set)
nulos_por_col = data.isnull().mean() * 100

# Filtrar únicamente las columnas que tienen 100% de valores nulos
cols_totalmente_nulas = nulos_por_col[nulos_por_col == 100.0].index.tolist()
print(f"Total de columnas 100% nulas: {len(cols_totalmente_nulas)}")
print(cols_totalmente_nulas)

# Eliminar todas las columnas que tengan 100% de nulos de forma automática
cols_a_borrar = data.columns[data.isnull().all()].tolist()
data = data.drop(columns=cols_a_borrar)

Total de columnas 100% nulas: 6
['antiguedad_meses_visa', 'antiguedad_meses_master', 'ctrx_quarter_lag6', 'mcuentas_saldo_lag6', 'ctrx_quarter_delta6', 'mcuentas_saldo_delta6']


In [24]:
data.shape

(983061, 229)

In [5]:
mes_train = 202103
mes_test = 202105

# agregue sus semillas
semilla = 42
semillas = np.random.choice(1000000, size=50, replace=False).tolist()

In [6]:
X = data[data['foto_mes'] == mes_train]
y = X['clase_ternaria']
X = X.drop(columns=['clase_ternaria'])

In [7]:
X_futuro = data[data['foto_mes'] == mes_test]
y_futuro = X_futuro['clase_ternaria']
X_futuro = X_futuro.drop(columns=['clase_ternaria'])

In [ ]:
#imp_median = SimpleImputer(missing_values=np.nan, strategy='median')
#Xi = imp_median.fit_transform(X)
#Xif = imp_median.fit_transform(X_futuro)
# No voy a imputar, tengo muchos nulos y a un compa le dio mejor dejandolo asi como esta, RF ahora los maneja

In [8]:
ganancia_acierto = 1_072_500
costo_estimulo = 27_500

def ganancia_prob(y_hat, y, prop=1, class_index=1, threshold=0.025):
  @np.vectorize
  def ganancia_row(predicted, actual, threshold=0.025):
    return  (predicted >= threshold) * (ganancia_acierto if actual == "BAJA+2" else -costo_estimulo)

  return ganancia_row(y_hat[:,class_index], y).sum() / prop

In [9]:
# 5 folds estratificados #Xi
from sklearn.model_selection import StratifiedKFold
n_splits = 5
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=semillas[0])

def objective(trial):
    max_depth = trial.suggest_int('max_depth', 2, 25)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 400, log=True)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 200, log=True)
    max_features = trial.suggest_float('max_features', 0.05, 0.7)
    max_samples = trial.suggest_float('max_samples', 0.4, 1.0)

    fold_scores = []

    for train_idx, val_idx in cv.split(X, y):
        X_train = X.iloc[train_idx] if hasattr(X, 'iloc') else X[train_idx]
        y_train = y.iloc[train_idx] if hasattr(y, 'iloc') else y[train_idx]
        X_val = X.iloc[val_idx] if hasattr(X, 'iloc') else X[val_idx]
        y_val = y.iloc[val_idx] if hasattr(y, 'iloc') else y[val_idx]

        model = RandomForestClassifier(
            n_estimators=200,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
            max_samples=max_samples,
            random_state=semillas[0],
            n_jobs=-1,
            oob_score=False
        )

        model.fit(X_train, y_train)

        # Matriz completa de probabilidades (n_muestras, n_clases)
        y_val_probs = model.predict_proba(X_val)

        # Detectar la columna exacta de 'BAJA+2' dinámicamente
        target_class_idx = list(model.classes_).index("BAJA+2")

        # Se ajusta prop al tamaño del fold relativo al total
        fold_gain = ganancia_prob(
            y_hat=y_val_probs,
            y=y_val,
            prop=1 / n_splits,
            class_index=target_class_idx
        )
        fold_scores.append(fold_gain)

    return float(np.mean(fold_scores))

In [30]:
storage_name = "sqlite:////Users/carof/Downloads/DENICOLAY/optimization_rf.db"
study_name = "exp_401_random-forest-opt-vFE13sept"

study = optuna.create_study(
    direction="maximize",
    study_name=study_name,
    storage=storage_name,
    load_if_exists=True,
)

[I 2026-09-13 14:17:41,182] Using an existing study with name 'exp_401_random-forest-opt-vFE13sept' instead of creating a new one.


In [ ]:
study.optimize(objective, n_trials=100)

[I 2026-09-13 14:28:11,527] Trial 1 finished with value: 338662500.0 and parameters: {'max_depth': 12, 'min_samples_split': 7, 'min_samples_leaf': 51, 'max_features': 0.6469832642813235, 'max_samples': 0.570706558563491}. Best is trial 1 with value: 338662500.0.
[I 2026-09-13 14:31:37,302] Trial 2 finished with value: 323675000.0 and parameters: {'max_depth': 6, 'min_samples_split': 63, 'min_samples_leaf': 76, 'max_features': 0.31541762933464373, 'max_samples': 0.7465247626783105}. Best is trial 1 with value: 338662500.0.
[I 2026-09-13 14:37:14,064] Trial 3 finished with value: 310447500.0 and parameters: {'max_depth': 9, 'min_samples_split': 53, 'min_samples_leaf': 6, 'max_features': 0.5241367695316642, 'max_samples': 0.4259144956071651}. Best is trial 1 with value: 338662500.0.
[I 2026-09-13 14:38:22,380] Trial 4 finished with value: 305745000.0 and parameters: {'max_depth': 6, 'min_samples_split': 212, 'min_samples_leaf': 10, 'max_features': 0.07711944721252117, 'max_samples': 0.941

In [10]:
# PARA SEGUIR EJECUTANDO SI SE DETUVO 

storage_name = "sqlite:////Users/carof/Downloads/DENICOLAY/optimization_rf.db"
study_name = "exp_401_random-forest-opt-vFE13sept"

# 1. Cargar el estudio existente
study = optuna.load_study(
    study_name=study_name,
    storage=storage_name,
)

# 2. Calcular cuántos trials faltan para llegar a la meta (ej. 100 trials totales)
TOTAL_TRIALS = 100
completed_trials = len(study.trials)
remaining_trials = max(0, TOTAL_TRIALS - completed_trials)

print(f"Trials completados: {completed_trials} | Restantes: {remaining_trials}")

# 3. Continuar la optimización si restan trials
if remaining_trials > 0:
    study.optimize(objective, n_trials=remaining_trials)

Trials completados: 49 | Restantes: 51


[I 2026-09-14 14:10:34,162] Trial 49 finished with value: 324280000.0 and parameters: {'max_depth': 17, 'min_samples_split': 2, 'min_samples_leaf': 46, 'max_features': 0.4521121154128448, 'max_samples': 0.9822964743632623}. Best is trial 35 with value: 348645000.0.
[I 2026-09-14 14:25:20,430] Trial 50 finished with value: 327910000.0 and parameters: {'max_depth': 12, 'min_samples_split': 14, 'min_samples_leaf': 23, 'max_features': 0.6455296089428854, 'max_samples': 0.8370413665759139}. Best is trial 35 with value: 348645000.0.
[I 2026-09-14 14:32:59,296] Trial 51 finished with value: 327387500.0 and parameters: {'max_depth': 20, 'min_samples_split': 94, 'min_samples_leaf': 85, 'max_features': 0.3292895534070893, 'max_samples': 0.7514506734177663}. Best is trial 35 with value: 348645000.0.
[I 2026-09-14 14:43:38,618] Trial 52 finished with value: 327085000.0 and parameters: {'max_depth': 15, 'min_samples_split': 244, 'min_samples_leaf': 139, 'max_features': 0.49780703240783464, 'max_sam

KeyboardInterrupt: 